[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARODBE/UAX_Fundamento_Ciencia_de_Datos/blob/main/Notebooks/Fase_2_Organizacion_Datos.ipynb)

# Fase 2. Organización y entendimiento de los datos

En esta fase preparo el conjunto de datos que usaré en las siguientes partes. Leo DILIst y una tabla compacta de asociaciones de nombres con estructuras. Calculo los descriptores moleculares de RDKit, reviso la calidad de las coincidencias y guardo los CSV de entrenamiento y prueba. La unidad de análisis será una estructura química identificada por su InChIKey.

Uso DILIst como fuente de etiquetas de lesión hepática. La tabla de estructuras reúne las asociaciones hechas con nombres preferidos de ChEMBL y búsquedas de respaldo en PubChem. Solo incorporo una estructura cuando la identificación es única por InChIKey. El archivo conserva también los nombres sin coincidencia inequívoca para que pueda revisar la cobertura.

In [1]:
# Instalo RDKit para calcular las propiedades moleculares.
%pip -q install rdkit

# Importo las herramientas que necesito para leer, preparar y guardar los datos.
from pathlib import Path  # Organizo las rutas de los archivos.
import pandas as pd  # Trabajo con tablas.
import numpy as np  # Uso valores numéricos y nulos.
from rdkit import Chem  # Convierto SMILES en moléculas.
from rdkit.Chem import Descriptors  # Accedo al catálogo amplio de descriptores RDKit.
from IPython.display import display  # Muestro tablas en Colab.

# Defino la carpeta de trabajo y la creo si todavía no existe.
DATA_DIR = Path('/content/datos_dili')  # Guardo los CSV en el espacio temporal de Colab.
DATA_DIR.mkdir(parents=True, exist_ok=True)  # Creo la carpeta para los archivos preparados.

# Defino enlaces públicos para ejecutar el notebook sin cargar archivos manualmente.
BASE_URL = 'https://raw.githubusercontent.com/PARODBE/UAX_Fundamento_Ciencia_de_Datos/main/Data/'  # Indico la carpeta pública.
URL_DILIST = BASE_URL + 'DILIst_FDA.csv'  # Indico el CSV de DILIst.
URL_ESTRUCTURAS = BASE_URL + 'estructuras_nombres.csv'  # Indico el mapa compacto de nombres y estructuras.


In [2]:
# Leo la fuente y el mapa de estructuras desde el repositorio del proyecto.
dilist_original = pd.read_csv(URL_DILIST)  # Cargo los registros originales de DILIst.
estructuras = pd.read_csv(URL_ESTRUCTURAS)  # Cargo las asociaciones de nombres y estructuras.

# Compruebo el tamaño de las tablas antes de combinarlas.
print('DILIst:', dilist_original.shape)  # Muestro filas y columnas de la fuente.
print('Mapa de estructuras:', estructuras.shape)  # Muestro filas y columnas del mapa.

# Reviso algunas filas para asegurarme de que los campos se hayan leído bien.
display(dilist_original.head())  # Muestro una muestra de DILIst.
display(estructuras.head())  # Muestro una muestra de las estructuras.

# Compruebo las dimensiones y los tipos de dato del archivo de la FDA.
print('Filas y columnas de DILIst:', dilist_original.shape)  # Informo el tamaño de la fuente.
display(dilist_original.dtypes.astype(str).rename('tipo de dato').to_frame())  # Muestro el tipo de cada columna.

# Describo las columnas que utilizaré en la preparación.
descripcion = pd.DataFrame({'variable': ['DILIST_ID', 'CompoundName', 'DILIst_Classification', 'Routes_of_Administration'], 'descripción': ['Identificador del registro', 'Nombre del compuesto', 'Etiqueta DILI: 0 negativa y 1 positiva', 'Vía de administración registrada en la fuente']})  # Creo una descripción breve.
display(descripcion)  # Muestro el diccionario de variables.

# Indico el contexto temporal y la unidad de análisis.
print('Unidad de análisis: un registro de fármaco de DILIst; uso InChIKey para identificar la estructura cuando está disponible.')  # Explico qué representa cada fila.
print('Rango temporal: no está disponible porque la fuente no contiene una columna de fecha por registro.')  # Aclaro la limitación temporal.


DILIst: (1279, 4)
Mapa de estructuras: (1279, 5)
Filas y columnas de DILIst: (1279, 4)
Unidad de análisis: un registro de fármaco de DILIst; uso InChIKey para identificar la estructura cuando está disponible.
Rango temporal: no está disponible porque la fuente no contiene una columna de fecha por registro.


,DILIST_ID,CompoundName,DILIst_Classification,Routes_of_Administration
0,1,mercaptopurine,1,Oral
1,2,acetaminophen,1,Oral
2,3,azathioprine,1,Oral
3,4,chlorpheniramine,0,Oral
4,5,clofibrate,1,Oral


,CompoundName,InChIKey,SMILES,source_estructura,estado_estructura
0,mercaptopurine,WFFQYWAAEWLHJC-UHFFFAOYSA-N,O.S=c1[nH]cnc2nc[nH]c12,ChEMBL,estructura única
1,acetaminophen,RZVAJINKPMORJF-UHFFFAOYSA-N,CC(=O)Nc1ccc(O)cc1,ChEMBL,estructura única
2,azathioprine,LMEKQMALGUDUQG-UHFFFAOYSA-N,Cn1cnc([N+](=O)[O-])c1Sc1ncnc2[nH]cnc12,ChEMBL,estructura única
3,chlorpheniramine,SOYKEARSMXGVTM-UHFFFAOYSA-N,CN(C)CCC(c1ccc(Cl)cc1)c1ccccn1,ChEMBL,estructura única
4,clofibrate,KNHUKKLJHYUCFP-UHFFFAOYSA-N,CCOC(=O)C(C)(C)Oc1ccc(Cl)cc1,ChEMBL,estructura única


,tipo de dato
DILIST_ID,int64
CompoundName,str
DILIst_Classification,int64
Routes_of_Administration,str


,variable,descripción
0,DILIST_ID,Identificador del registro
1,CompoundName,Nombre del compuesto
2,DILIst_Classification,Etiqueta DILI: 0 negativa y 1 positiva
3,Routes_of_Administration,Vía de administración registrada en la fuente


Normalizo los nombres solo para hacer la comparación, pero conservo el nombre original de DILIst en la tabla final. El mapa de estructuras ya incluye una fila por nombre y señala si la estructura quedó identificada, ambigua o sin coincidencia.

In [3]:
# Renombro la etiqueta para trabajar con un nombre corto y uniforme.
dilist = dilist_original.rename(columns={'DILIst_Classification': 'DILI'}).copy()  # Copio los datos y renombro la clase.
dilist['CompoundName'] = dilist['CompoundName'].astype(str).str.strip()  # Quito espacios sobrantes del nombre.
dilist['DILI'] = pd.to_numeric(dilist['DILI'], errors='coerce')  # Convierto la etiqueta a número.
dilist = dilist.dropna(subset=['CompoundName', 'DILI']).copy()  # Retiro filas sin nombre o clase.

# Preparo claves normalizadas en las dos tablas para unir nombres sin distinguir mayúsculas.
dilist['nombre_clave'] = dilist['CompoundName'].str.upper()  # Normalizo nombres de DILIst.
estructuras['nombre_clave'] = estructuras['CompoundName'].astype(str).str.strip().str.upper()  # Normalizo nombres del mapa.
estructuras = estructuras.drop(columns='CompoundName')  # Quito el nombre duplicado del mapa.
datos = dilist.merge(estructuras, on='nombre_clave', how='left')  # Uno las etiquetas y las estructuras.
datos = datos.drop(columns='nombre_clave')  # Quito la clave auxiliar de trabajo.

# Reviso cuántas filas tienen una estructura inequívoca.
print('Registros DILIst:', len(datos))  # Muestro el total de registros.
print('Con InChIKey:', datos['InChIKey'].notna().sum())  # Cuento las estructuras resueltas.
print('Sin estructura inequívoca:', datos['InChIKey'].isna().sum())  # Cuento los casos pendientes.


Registros DILIst: 1279
Con InChIKey: 1186
Sin estructura inequívoca: 93


Calculo todos los descriptores escalares que trae el catálogo de RDKit (en esta versión, alrededor de 200). Así no reduzco de antemano la información química disponible. Para el EDA describo cinco propiedades fáciles de interpretar —las cuatro de Lipinski y TPSA—; en los modelos selecciono entre el conjunto amplio y retiro variables constantes o muy correlacionadas usando solo entrenamiento. No calculo fingerprints.

In [4]:
# Defino una función que calcula todos los descriptores escalares disponibles en RDKit.
def calcular_descriptores(smiles):  # Recibo una cadena SMILES.
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles else None  # Creo la molécula si hay SMILES.
    if mol is None:  # Compruebo que RDKit haya leído la estructura.
        return {}  # Devuelvo vacío si no puedo procesarla.
    valores = {}  # Creo un diccionario para las propiedades moleculares.
    for nombre, funcion in Descriptors._descList:  # Recorro el catálogo de descriptores RDKit.
        try:  # Intento calcular cada propiedad.
            valores[nombre] = funcion(mol)  # Guardo el resultado calculado.
        except Exception:  # Evito que un descriptor aislado detenga todo el cálculo.
            valores[nombre] = np.nan  # Marco como ausente el valor que no pude calcular.
    return valores  # Devuelvo todos los descriptores calculados.

# Calculo una fila de propiedades para cada SMILES.
lista_descriptores = [calcular_descriptores(smiles) for smiles in datos['SMILES']]  # Aplico la función a las estructuras.
descriptores = pd.DataFrame(lista_descriptores)  # Convierto los resultados en una tabla.
datos = pd.concat([datos.reset_index(drop=True), descriptores], axis=1)  # Uno los descriptores con las etiquetas.

# Cuento incumplimientos de los cuatro límites de Lipinski.
datos['LipinskiViolations'] = ((datos['MolWt'] >= 500).astype(int) + (datos['MolLogP'] > 5).astype(int) + (datos['NumHDonors'] > 5).astype(int) + (datos['NumHAcceptors'] > 10).astype(int))  # Sumo los cuatro criterios.

# Muestro algunas variables químicas para revisar el cálculo.
columnas_demo = ['CompoundName', 'DILI', 'InChIKey', 'MolWt', 'MolLogP', 'NumHDonors', 'NumHAcceptors', 'TPSA']  # Selecciono columnas legibles.
display(datos[columnas_demo].head())  # Muestro una muestra de los resultados.
print('Descriptores RDKit calculados:', len(descriptores.columns))  # Informo cuántas propiedades se calcularon.


Descriptores RDKit calculados: 217


,CompoundName,DILI,InChIKey,MolWt,MolLogP,NumHDonors,NumHAcceptors,TPSA
0,mercaptopurine,1,WFFQYWAAEWLHJC-UHFFFAOYSA-N,170.197,0.19079,2.0,3.0,88.86
1,acetaminophen,1,RZVAJINKPMORJF-UHFFFAOYSA-N,151.165,1.35060,2.0,2.0,49.33
2,azathioprine,1,LMEKQMALGUDUQG-UHFFFAOYSA-N,277.269,1.14580,1.0,7.0,115.42
3,chlorpheniramine,0,SOYKEARSMXGVTM-UHFFFAOYSA-N,274.795,3.81860,0.0,2.0,16.13
4,clofibrate,1,KNHUKKLJHYUCFP-UHFFFAOYSA-N,242.702,3.06050,0.0,3.0,35.53


## 2.4. Calidad, duplicados y partición

Retiro de los análisis moleculares los registros sin estructura inequívoca. Si la misma InChIKey aparece con etiquetas DILI diferentes, la separo para revisión. Después dejo una fila por InChIKey y creo una partición 80/20. Estratifico por cuantiles de MolLogP y, cuando el tamaño de los grupos lo permite, también por la clase DILI.

In [5]:
# Normalizo la InChIKey y separo los registros con estructura identificada.
datos['InChIKey'] = datos['InChIKey'].astype('string').str.upper()  # Normalizo el identificador químico.
con_etiqueta = datos.dropna(subset=['InChIKey']).copy()  # Conservo moléculas con estructura.

# Identifico claves químicas que aparecen con etiquetas distintas.
clases_por_clave = con_etiqueta.groupby('InChIKey')['DILI'].nunique()  # Cuento etiquetas distintas por estructura.
claves_conflictivas = clases_por_clave[clases_por_clave > 1].index  # Selecciono estructuras con conflicto.
datos_conflicto = con_etiqueta[con_etiqueta['InChIKey'].isin(claves_conflictivas)].copy()  # Guardo esos casos aparte.
datos_limpios = con_etiqueta[~con_etiqueta['InChIKey'].isin(claves_conflictivas)].drop_duplicates('InChIKey').reset_index(drop=True)  # Dejo una fila por estructura válida.

# Busco un número de cuantiles que permita estratificar sin grupos demasiado pequeños.
estratos = None  # Inicio sin una estratificación elegida.
for numero_cuantiles in range(10, 1, -1):  # Pruebo de diez cuantiles a dos.
    cuantiles = pd.qcut(datos_limpios['MolLogP'], q=numero_cuantiles, duplicates='drop')  # Agrupo MolLogP por cuantiles.
    propuesta = datos_limpios['DILI'].astype(int).astype(str) + '_' + cuantiles.astype(str)  # Combino cuantiles y clase.
    if propuesta.value_counts().min() >= 2:  # Compruebo que cada grupo permita dividirse.
        estratos = propuesta  # Guardo la estratificación más detallada disponible.
        break  # Termino la búsqueda.
if estratos is None:  # Compruebo si no fue posible combinar con la clase.
    estratos = cuantiles if cuantiles.value_counts().min() >= 2 else None  # Uso solo cuantiles si es posible.

# Creo la partición fija para que los resultados sean reproducibles.
from sklearn.model_selection import train_test_split  # Importo la función de partición.
train_df, test_df = train_test_split(datos_limpios, test_size=0.20, random_state=42, stratify=estratos)  # Separo 80 % y 20 %.
print('Train:', train_df.shape, '| Test:', test_df.shape)  # Muestro el tamaño de cada parte.
print('Clases en train:', train_df['DILI'].value_counts(normalize=True).round(3).to_dict())  # Reviso la proporción de clases en train.
print('Clases en test:', test_df['DILI'].value_counts(normalize=True).round(3).to_dict())  # Reviso la proporción de clases en test.


Train: (945, 226) | Test: (237, 226)
Clases en train: {1: 0.618, 0: 0.382}
Clases en test: {1: 0.616, 0: 0.384}


## 2.5. Exportación de los CSV

Guardo las tablas en la carpeta de trabajo de Colab. Las fases siguientes también pueden leer las versiones públicas de estos CSV desde GitHub, de modo que no dependen de una carga manual de archivos.

In [6]:
# Defino el conjunto de descriptores numéricos calculados, salvo columnas de identificación y resultado.
columnas_excluir = ['DILI', 'DILIST_ID', 'InChIKey', 'SMILES', 'CompoundName', 'Routes_of_Administration', 'chembl_id']  # Evito incluir etiquetas e identificadores como propiedades.
DESCRIPTORES = [columna for columna in descriptores.select_dtypes(include='number').columns if columna not in columnas_excluir]  # Selecciono propiedades numéricas.
DESCRIPTORES = [columna for columna in DESCRIPTORES if columna in datos_limpios.columns]  # Conservo solo columnas presentes en los datos depurados.

# Guardo únicamente las tablas que utilizaré en las fases siguientes.
train_df.to_csv(DATA_DIR / 'dilist_train.csv', index=False)  # Guardo la partición de entrenamiento.
test_df.to_csv(DATA_DIR / 'dilist_test.csv', index=False)  # Guardo la partición de prueba.
datos_limpios.to_csv(DATA_DIR / 'dilist_descriptores.csv', index=False)  # Guardo los datos moleculares depurados.

# Informo dónde quedaron los archivos preparados.
print('He guardado los CSV en:', DATA_DIR)  # Muestro la carpeta de salida.
print('Descriptores numéricos disponibles para los modelos:', len(DESCRIPTORES))  # Muestro el número de propiedades disponibles.


He guardado los CSV en: /workspace/scratch/b597cb51ba0f/run_outputs/datos_dili
Descriptores numéricos disponibles para los modelos: 217


Al terminar esta fase puedo comprobar cuántos registros tienen una estructura asociada, cuántos quedan fuera por falta de una coincidencia inequívoca y cuántos compuestos entran en cada partición. Los modelos podrán considerar el conjunto amplio de descriptores; en la fase exploratoria mostraré solo un grupo reducido para que los resultados se puedan interpretar.